# Домашнее задание № 4. Языковые модели

## Задание 1 (8 баллов).

В семинаре для генерации мы использовали предположение маркова и считали, что слово зависит только от 1 предыдущего слова. Но ничто нам не мешает попробовать увеличить размер окна и учитывать два или даже три прошлых слова. Для них мы еще сможем собрать достаточно статистик и, логично предположить, что качество сгенерированного текста должно вырасти.

Попробуйте сделать языковую модель, которая будет учитывать два предыдущих слова при генерации текста.
Сгенерируйте несколько текстов (3-5) и расчитайте перплексию получившейся модели.
Можно использовать данные из семинара или любые другие (можно брать только часть текста, если считается слишком долго). Перплексию рассчитывайте на 10-50 отложенных предложениях (они не должны использоваться при сборе статистик).


Подсказки:  
    - нужно будет добавить еще один тэг \<start>  
    - можете использовать тот же подход с матрицей вероятностей, но по строкам хронить биграмы, а по колонкам униграммы
    - тексты должны быть очень похожи на нормальные (если у вас получается рандомная каша, вы что-то делаете не так)
    - у вас будут словари с индексами биграммов и униграммов, не перепутайте их при переводе индекса в слово - словарь биграммов будет больше словаря униграммов и все индексы из униграммного словаря будут формально подходить для словаря биграммов (не будет ошибки при id2bigram[unigram_id]), но маппинг при этом будет совершенно неправильным

In [1]:
import numpy as np
import re
from collections import defaultdict
import math
import random

class TrigramLanguageModel:
    def __init__(self):
        self.unigram_vocab = {}
        self.bigram_vocab = {}
        self.trigram_counts = defaultdict(int)
        self.bigram_counts = defaultdict(int)
        self.unigram_counts = defaultdict(int)
        self.smoothing = 0.1

    def build_vocab(self, texts):
        all_words = set()
        for text in texts:
            words = self._tokenize(text)
            all_words.update(words)

        all_words.add('<start1>')
        all_words.add('<start2>')
        all_words.add('<end>')

        self.unigram_vocab = {word: idx for idx, word in enumerate(sorted(all_words))}
        self.id2unigram = {idx: word for word, idx in self.unigram_vocab.items()}

        bigrams = set()
        for text in texts:
            words = self._tokenize(text)
            if len(words) < 2:
                continue

            bigrams.add(('<start1>', '<start2>'))
            bigrams.add(('<start2>', words[0]))

            for i in range(len(words) - 1):
                bigrams.add((words[i], words[i+1]))

            bigrams.add((words[-1], '<end>'))

        self.bigram_vocab = {bigram: idx for idx, bigram in enumerate(sorted(bigrams))}
        self.id2bigram = {idx: bigram for bigram, idx in self.bigram_vocab.items()}

    def _tokenize(self, text):
        text = text.lower()
        text = re.sub(r'[^\w\s]', ' ', text)
        words = text.split()
        return words

    def train(self, texts):
        self.build_vocab(texts)

        for text in texts:
            words = self._tokenize(text)
            if len(words) < 2:
                continue

            bigram_id = self.bigram_vocab[('<start1>', '<start2>')]
            unigram_id = self.unigram_vocab[words[0]]
            self.trigram_counts[(bigram_id, unigram_id)] += 1
            self.bigram_counts[bigram_id] += 1
            self.unigram_counts[unigram_id] += 1

            bigram_id = self.bigram_vocab[('<start2>', words[0])]
            unigram_id = self.unigram_vocab[words[1]]
            self.trigram_counts[(bigram_id, unigram_id)] += 1
            self.bigram_counts[bigram_id] += 1
            self.unigram_counts[unigram_id] += 1

            for i in range(len(words) - 2):
                bigram = (words[i], words[i+1])
                next_word = words[i+2]

                bigram_id = self.bigram_vocab[bigram]
                unigram_id = self.unigram_vocab[next_word]

                self.trigram_counts[(bigram_id, unigram_id)] += 1
                self.bigram_counts[bigram_id] += 1
                self.unigram_counts[unigram_id] += 1

            if len(words) >= 2:
                last_bigram = (words[-2], words[-1])
                bigram_id = self.bigram_vocab[last_bigram]
                unigram_id = self.unigram_vocab['<end>']
                self.trigram_counts[(bigram_id, unigram_id)] += 1
                self.bigram_counts[bigram_id] += 1
                self.unigram_counts[unigram_id] += 1

    def get_probability(self, bigram_id, unigram_id):
        """Вероятность слова given биграмма"""
        count_trigram = self.trigram_counts.get((bigram_id, unigram_id), 0)
        count_bigram = self.bigram_counts.get(bigram_id, 0)

        vocab_size = len(self.unigram_vocab)
        probability = (count_trigram + self.smoothing) / (count_bigram + self.smoothing * vocab_size)
        return probability

    def get_fallback_probability(self, word):
        """Резервная вероятность на основе униграмм"""
        if word in self.unigram_vocab:
            unigram_id = self.unigram_vocab[word]
            total_words = sum(self.unigram_counts.values())
            return (self.unigram_counts[unigram_id] + self.smoothing) / (total_words + self.smoothing * len(self.unigram_vocab))
        return self.smoothing / (sum(self.unigram_counts.values()) + self.smoothing * len(self.unigram_vocab))

    def generate_text(self, max_length=20):
        """Генерация текста с обработкой неизвестных биграмм"""
        words = ['<start1>', '<start2>']
        bigram = ('<start1>', '<start2>')

        for _ in range(max_length):
            if bigram not in self.bigram_vocab:
                if bigram[1] in self.unigram_vocab:
                    last_word = bigram[1]
                    possible_words = []
                    probabilities = []

                    for word, unigram_id in self.unigram_vocab.items():
                        if word not in ['<start1>', '<start2>', '<end>']:
                            prob = self.get_fallback_probability(word)
                            probabilities.append(prob)
                            possible_words.append(word)
                else:

                    possible_words = [word for word in self.unigram_vocab.keys()
                                    if word not in ['<start1>', '<start2>', '<end>']]
                    probabilities = [1.0/len(possible_words)] * len(possible_words)
            else:
                bigram_id = self.bigram_vocab[bigram]

                probabilities = []
                possible_words = []

                for word, unigram_id in self.unigram_vocab.items():
                    if word not in ['<start1>', '<start2>']:
                        prob = self.get_probability(bigram_id, unigram_id)
                        probabilities.append(prob)
                        possible_words.append(word)

            total_prob = sum(probabilities)
            if total_prob == 0:
                probabilities = [1.0/len(possible_words)] * len(possible_words)
            else:
                probabilities = [p / total_prob for p in probabilities]

            chosen_idx = np.random.choice(len(possible_words), p=probabilities)
            next_word = possible_words[chosen_idx]

            if next_word == '<end>':
                break

            words.append(next_word)
            bigram = (bigram[1], next_word)

        return ' '.join(words[2:])

    def calculate_perplexity(self, test_texts):
        total_log_prob = 0
        total_words = 0

        for text in test_texts:
            words = self._tokenize(text)
            if len(words) < 3:
                continue

            for i in range(2, len(words)):
                bigram = (words[i-2], words[i-1])
                next_word = words[i]

                prob = 0
                if bigram in self.bigram_vocab and next_word in self.unigram_vocab:
                    bigram_id = self.bigram_vocab[bigram]
                    unigram_id = self.unigram_vocab[next_word]
                    prob = self.get_probability(bigram_id, unigram_id)
                else:
                    prob = self.get_fallback_probability(next_word)

                if prob > 0:
                    total_log_prob += math.log(prob)
                    total_words += 1

        if total_words == 0:
            return float('inf')

        avg_log_prob = total_log_prob / total_words
        perplexity = math.exp(-avg_log_prob)
        return perplexity

texts = [
    "the cat sat on the mat",
    "the dog ran in the park",
    "a cat and a dog played together",
    "the sun shines bright today",
    "birds fly in the sky",
    "children play in the garden",
    "I love to read books",
    "she sings a beautiful song",
    "we go to school every day",
    "he eats an apple for breakfast",
    "the weather is nice today",
    "they watch television at night",
    "my friend lives in a big house",
    "the car drives on the road",
    "flowers bloom in spring",
    "water flows down the river",
    "the teacher explains the lesson",
    "students learn new things",
    "computers help with work",
    "music makes people happy"
]


train_texts = texts[:15]
test_texts = texts[15:]


model = TrigramLanguageModel()
model.train(train_texts)

print("Сгенерированные тексты:")
print("-" * 50)
for i in range(5):
    generated_text = model.generate_text(max_length=8)
    print(f"{i+1}. {generated_text}")

perplexity = model.calculate_perplexity(test_texts)
print(f"\nПерплексия на тестовых данных: {perplexity:.2f}")

print(f"\nРазмер словаря униграмм: {len(model.unigram_vocab)}")
print(f"Размер словаря биграмм: {len(model.bigram_vocab)}")

Сгенерированные тексты:
--------------------------------------------------
1. the for sings lives i spring go eats
2. today for day the in big the weather
3. for a nice garden friend on big she
4. the weather car in for park car children
5. a for the played in my the play

Перплексия на тестовых данных: 484.24

Размер словаря униграмм: 65
Размер словаря биграмм: 91


## Задание № 2* (2 балла).

Измените функцию generate_with_beam_search так, чтобы она работала с моделью, которая учитывает два предыдущих слова.
Сравните получаемый результат с первым заданием.
Также попробуйте начинать генерацию не с нуля (подавая \<start> \<start>), а с какого-то промпта. Но помните, что учитываться будут только два последних слова, так что не делайте длинные промпты.

In [7]:
import numpy as np
import re
from collections import defaultdict
import math
import heapq

class TrigramLanguageModel:
    def __init__(self):
        self.unigram_vocab = {}
        self.bigram_vocab = {}
        self.trigram_counts = defaultdict(int)
        self.bigram_counts = defaultdict(int)
        self.unigram_counts = defaultdict(int)
        self.smoothing = 0.1

    def build_vocab(self, texts):
        all_words = set()
        for text in texts:
            words = self._tokenize(text)
            all_words.update(words)

        all_words.add('<start1>')
        all_words.add('<start2>')
        all_words.add('<end>')

        self.unigram_vocab = {word: idx for idx, word in enumerate(sorted(all_words))}
        self.id2unigram = {idx: word for word, idx in self.unigram_vocab.items()}

        bigrams = set()
        for text in texts:
            words = self._tokenize(text)
            if len(words) < 2:
                continue

            bigrams.add(('<start1>', '<start2>'))
            bigrams.add(('<start2>', words[0]))

            for i in range(len(words) - 1):
                bigrams.add((words[i], words[i+1]))

            bigrams.add((words[-1], '<end>'))

        self.bigram_vocab = {bigram: idx for idx, bigram in enumerate(sorted(bigrams))}
        self.id2bigram = {idx: bigram for bigram, idx in self.bigram_vocab.items()}

    def _tokenize(self, text):
        text = text.lower()
        text = re.sub(r'[^\w\s]', ' ', text)
        words = text.split()
        return words

    def train(self, texts):
        self.build_vocab(texts)

        for text in texts:
            words = self._tokenize(text)
            if len(words) < 2:
                continue

            bigram_id = self.bigram_vocab[('<start1>', '<start2>')]
            unigram_id = self.unigram_vocab[words[0]]
            self.trigram_counts[(bigram_id, unigram_id)] += 1
            self.bigram_counts[bigram_id] += 1
            self.unigram_counts[unigram_id] += 1

            bigram_id = self.bigram_vocab[('<start2>', words[0])]
            unigram_id = self.unigram_vocab[words[1]]
            self.trigram_counts[(bigram_id, unigram_id)] += 1
            self.bigram_counts[bigram_id] += 1
            self.unigram_counts[unigram_id] += 1

            for i in range(len(words) - 2):
                bigram = (words[i], words[i+1])
                next_word = words[i+2]

                bigram_id = self.bigram_vocab[bigram]
                unigram_id = self.unigram_vocab[next_word]

                self.trigram_counts[(bigram_id, unigram_id)] += 1
                self.bigram_counts[bigram_id] += 1
                self.unigram_counts[unigram_id] += 1

            if len(words) >= 2:
                last_bigram = (words[-2], words[-1])
                bigram_id = self.bigram_vocab[last_bigram]
                unigram_id = self.unigram_vocab['<end>']
                self.trigram_counts[(bigram_id, unigram_id)] += 1
                self.bigram_counts[bigram_id] += 1
                self.unigram_counts[unigram_id] += 1

    def get_probability(self, bigram_id, unigram_id):
        """Вероятность слова given биграмма"""
        count_trigram = self.trigram_counts.get((bigram_id, unigram_id), 0)
        count_bigram = self.bigram_counts.get(bigram_id, 0)

        vocab_size = len(self.unigram_vocab)
        probability = (count_trigram + self.smoothing) / (count_bigram + self.smoothing * vocab_size)
        return probability

    def get_fallback_probability(self, word):
        """Резервная вероятность на основе униграмм"""
        if word in self.unigram_vocab:
            unigram_id = self.unigram_vocab[word]
            total_words = sum(self.unigram_counts.values())
            return (self.unigram_counts[unigram_id] + self.smoothing) / (total_words + self.smoothing * len(self.unigram_vocab))
        return self.smoothing / (sum(self.unigram_counts.values()) + self.smoothing * len(self.unigram_vocab))

    def generate_text(self, max_length=20):
        """Генерация текста с обработкой неизвестных биграмм"""
        words = ['<start1>', '<start2>']
        bigram = ('<start1>', '<start2>')

        for _ in range(max_length):
            if bigram not in self.bigram_vocab:
                if bigram[1] in self.unigram_vocab:
                    last_word = bigram[1]
                    possible_words = []
                    probabilities = []

                    for word, unigram_id in self.unigram_vocab.items():
                        if word not in ['<start1>', '<start2>', '<end>']:
                            prob = self.get_fallback_probability(word)
                            probabilities.append(prob)
                            possible_words.append(word)
                else:
                    possible_words = [word for word in self.unigram_vocab.keys()
                                    if word not in ['<start1>', '<start2>', '<end>']]
                    probabilities = [1.0/len(possible_words)] * len(possible_words)
            else:
                bigram_id = self.bigram_vocab[bigram]

                probabilities = []
                possible_words = []

                for word, unigram_id in self.unigram_vocab.items():
                    if word not in ['<start1>', '<start2>']:
                        prob = self.get_probability(bigram_id, unigram_id)
                        probabilities.append(prob)
                        possible_words.append(word)

            total_prob = sum(probabilities)
            if total_prob == 0:
                probabilities = [1.0/len(possible_words)] * len(possible_words)
            else:
                probabilities = [p / total_prob for p in probabilities]

            chosen_idx = np.random.choice(len(possible_words), p=probabilities)
            next_word = possible_words[chosen_idx]

            if next_word == '<end>':
                break

            words.append(next_word)
            bigram = (bigram[1], next_word)

        return ' '.join(words[2:])

    def _prepare_prompt_context(self, prompt):
        """Подготавливает контекст из промпта"""
        prompt_words = self._tokenize(prompt)

        if len(prompt_words) == 0:
            return [('<start1>', '<start2>')], ['<start1>', '<start2>']
        elif len(prompt_words) == 1:
            return [('<start2>', prompt_words[0])], ['<start1>', '<start2>', prompt_words[0]]
        else:
            last_bigram = (prompt_words[-2], prompt_words[-1])
            return [last_bigram], prompt_words

    def generate_with_beam_search(self, beam_width=3, max_length=15, prompt=None):
        """
        Генерация текста с использованием beam search
        """
        if prompt is None:
            sequences = [([('<start1>', '<start2>')], 0.0)]
            context_words = ['<start1>', '<start2>']
        else:
            sequences, context_words = self._prepare_prompt_context(prompt)
            sequences = [(sequences, 0.0)]

        for step in range(max_length):
            all_candidates = []

            for seq, score in sequences:
                last_bigram = seq[-1]

                if last_bigram not in self.bigram_vocab:
                    for word in self.unigram_vocab:
                        if word not in ['<start1>', '<start2>', '<end>']:
                            prob = self.get_fallback_probability(word)
                            if prob > 0:
                                new_bigram = (last_bigram[1], word)
                                new_seq = seq + [new_bigram]
                                new_score = score + math.log(prob)
                                all_candidates.append((new_seq, new_score))
                else:
                    bigram_id = self.bigram_vocab[last_bigram]

                    for word, unigram_id in self.unigram_vocab.items():
                        if word not in ['<start1>', '<start2>']:
                            prob = self.get_probability(bigram_id, unigram_id)
                            if prob > 0:
                                new_bigram = (last_bigram[1], word)
                                new_seq = seq + [new_bigram]
                                new_score = score + math.log(prob)
                                all_candidates.append((new_seq, new_score))

                if last_bigram in self.bigram_vocab:
                    bigram_id = self.bigram_vocab[last_bigram]
                    end_prob = self.get_probability(bigram_id, self.unigram_vocab['<end>'])
                    if end_prob > 0:
                        all_candidates.append((seq, score + math.log(end_prob)))

            if not all_candidates:
                break

            all_candidates.sort(key=lambda x: x[1], reverse=True)
            sequences = all_candidates[:beam_width]

            completed = all(len(seq) > 0 and (seq[-1][1] == '<end>' or any(bigram[1] == '<end>' for bigram in seq))
                          for seq, score in sequences)
            if completed:
                break

        if sequences:
            best_seq, best_score = max(sequences, key=lambda x: x[1])

            if prompt is None:
                words = [bigram[1] for bigram in best_seq[1:]]
            else:
                if len(context_words) >= 2:
                    words = context_words[:-2] if len(context_words) > 2 else []

                    words.extend([bigram[1] for bigram in best_seq])
                else:
                    words = [bigram[1] for bigram in best_seq[1:]]

            words = [w for w in words if w != '<end>']
            return ' '.join(words), best_score
        else:
            return "", -float('inf')

    def calculate_perplexity(self, test_texts):
        total_log_prob = 0
        total_words = 0

        for text in test_texts:
            words = self._tokenize(text)
            if len(words) < 3:
                continue

            for i in range(2, len(words)):
                bigram = (words[i-2], words[i-1])
                next_word = words[i]

                prob = 0
                if bigram in self.bigram_vocab and next_word in self.unigram_vocab:
                    bigram_id = self.bigram_vocab[bigram]
                    unigram_id = self.unigram_vocab[next_word]
                    prob = self.get_probability(bigram_id, unigram_id)
                else:
                    prob = self.get_fallback_probability(next_word)

                if prob > 0:
                    total_log_prob += math.log(prob)
                    total_words += 1

        if total_words == 0:
            return float('inf')

        avg_log_prob = total_log_prob / total_words
        perplexity = math.exp(-avg_log_prob)
        return perplexity

texts = [
    "the cat sat on the mat",
    "the dog ran in the park",
    "a cat and a dog played together",
    "the sun shines bright today",
    "birds fly in the sky",
    "children play in the garden",
    "I love to read books",
    "she sings a beautiful song",
    "we go to school every day",
    "he eats an apple for breakfast",
    "the weather is nice today",
    "they watch television at night",
    "my friend lives in a big house",
    "the car drives on the road",
    "flowers bloom in spring",
    "water flows down the river",
    "the teacher explains the lesson",
    "students learn new things",
    "computers help with work",
    "music makes people happy"
]

train_texts = texts[:15]
test_texts = texts[15:]

model = TrigramLanguageModel()
model.train(train_texts)

print("=== СРАВНЕНИЕ МЕТОДОВ ГЕНЕРАЦИИ ===\n")

print("1. ОБЫЧНАЯ ГЕНЕРАЦИЯ (случайный выбор):")
print("-" * 50)
for i in range(3):
    generated_text = model.generate_text(max_length=8)
    print(f"{i+1}. {generated_text}")

print("\n2. BEAM SEARCH ГЕНЕРАЦИЯ (beam_width=3):")
print("-" * 50)
for i in range(3):
    generated_text, score = model.generate_with_beam_search(beam_width=3, max_length=8)
    print(f"{i+1}. {generated_text} (score: {score:.2f})")

print("\n3. ГЕНЕРАЦИЯ С ПРОМПТАМИ:")
print("-" * 50)
prompts = ["the cat", "I love", "students learn", "weather is"]
for prompt in prompts:
    generated_text, score = model.generate_with_beam_search(beam_width=3, max_length=6, prompt=prompt)
    print(f"Промпт '{prompt}': {generated_text} (score: {score:.2f})")

print("\n=== АНАЛИЗ КАЧЕСТВА ГЕНЕРАЦИИ ===")

print("\nСравнение повторяемости:")
print("Обычная генерация (3 запуска):")
for i in range(3):
    print(f"  {model.generate_text(max_length=6)}")

print("Beam search (3 запуска):")
for i in range(3):
    text, score = model.generate_with_beam_search(beam_width=3, max_length=6)
    print(f"  {text}")

print("\nТест на осмысленность с разными промптами:")
test_prompts = ["the cat", "I love", "we go", "she sings", "they watch"]
for prompt in test_prompts:
    random_text = model.generate_text(max_length=6)
    beam_text, score = model.generate_with_beam_search(beam_width=3, max_length=6, prompt=prompt)
    print(f"Промпт '{prompt}':")
    print(f"  Случайная: {random_text}")
    print(f"  Beam: {beam_text}")

# Перплексия
perplexity = model.calculate_perplexity(test_texts)
print(f"\nПерплексия на тестовых данных: {perplexity:.2f}")

=== СРАВНЕНИЕ МЕТОДОВ ГЕНЕРАЦИИ ===

1. ОБЫЧНАЯ ГЕНЕРАЦИЯ (случайный выбор):
--------------------------------------------------
1. we a for school sky the a fly
2. we night together is my and garden the
3. he played a a ran in dog lives

2. BEAM SEARCH ГЕНЕРАЦИЯ (beam_width=3):
--------------------------------------------------
1. the car drives on the mat (score: -15.43)
2. the car drives on the mat (score: -15.43)
3. the car drives on the mat (score: -15.43)

3. ГЕНЕРАЦИЯ С ПРОМПТАМИ:
--------------------------------------------------
Промпт 'the cat': cat sat on the mat (score: -11.64)
Промпт 'I love': love to read books (score: -11.52)
Промпт 'students learn': learn in the garden (score: -13.27)
Промпт 'weather is': is nice today (score: -11.52)

=== АНАЛИЗ КАЧЕСТВА ГЕНЕРАЦИИ ===

Сравнение повторяемости:
Обычная генерация (3 запуска):
  i love to day a in
  i is apple park books sky
  i television cat today in apple
Beam search (3 запуска):
  the car drives on the mat
  the car dr

In [8]:

print("\n=== АНАЛИЗ КАЧЕСТВА ГЕНЕРАЦИИ ===")


print("\nСравнение повторяемости:")
print("Обычная генерация (3 запуска):")
for i in range(3):
    print(f"  {model.generate_text(max_length=6)}")

print("Beam search (3 запуска):")
for i in range(3):
    text, score = model.generate_with_beam_search(beam_width=3, max_length=6)
    print(f"  {text}")


print("\nТест на осмысленность с разными промптами:")
test_prompts = ["the", "I", "we", "she", "they"]
for prompt in test_prompts:
    random_text = model.generate_text(max_length=6)
    beam_text, score = model.generate_with_beam_search(beam_width=3, max_length=6, prompt=prompt)
    print(f"Промпт '{prompt}':")
    print(f"  Случайная: {random_text}")
    print(f"  Beam: {beam_text}")


=== АНАЛИЗ КАЧЕСТВА ГЕНЕРАЦИИ ===

Сравнение повторяемости:
Обычная генерация (3 запуска):
  he for to love at in
  she watch they together my love
  my spring song dog sings song
Beam search (3 запуска):
  the car drives on the mat
  the car drives on the mat
  the car drives on the mat

Тест на осмысленность с разными промптами:
Промпт 'the':
  Случайная: the sky day house in shines
  Beam: <start1> the car drives on the mat
Промпт 'I':
  Случайная: my drives spring to the we
  Beam: <start1> i love to read books
Промпт 'we':
  Случайная: my watch and read the on
  Beam: <start1> we go to school every day
Промпт 'she':
  Случайная: he house breakfast night she flowers
  Beam: <start1> she sings a beautiful song
Промпт 'they':
  Случайная: children in night sat the night
  Beam: <start1> they watch television at night
